# Detective Rice 😼 — Bangladeshi Rice Variety Recognition

Compare a close-up of one uncooked rice grain against six cultivar codes.

Dataset: 38 classes, 500 original microscope photos per class and 2,000 supplied augmentations per class. The experiment uses only originals, excludes all supplied augmentations, and applies new augmentation only during training. The archive provides no physical sample or batch IDs. Image-family grouping reduces duplicate leakage without establishing batch independence.

The model uses EfficientNet-B0 at 224 × 224. The held-out report measures microscope-image performance. The fresh phone-photo evaluation requires separately labeled batches.

In [ ]:
from collections import Counter
from pathlib import Path
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
from random import Random

input_root = Path("/kaggle/input")
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
folders = {}
for path in sorted(input_root.rglob("*")):
    if path.is_file() and path.suffix.lower() in image_extensions:
        folders.setdefault(path.parent, []).append(path)
print("Image folders:", len(folders), "| Images:", sum(map(len, folders.values())))
for folder, images in folders.items():
    print(f"{len(images):5}  {folder.relative_to(input_root)} | {images[0].name} … {images[-1].name}")
print("Metadata:", [str(p.relative_to(input_root)) for p in input_root.rglob("*")
                    if p.is_file() and p.suffix.lower() in {".csv", ".txt", ".json", ".xlsx"}][:30])
sample_folders = list(folders)
rng = Random(42)
for offset in range(0, len(sample_folders), 24):
    panel = sample_folders[offset:offset+24]
    fig, axes = plt.subplots((len(panel)+5)//6, 6, figsize=(14, 2.4*((len(panel)+5)//6)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, folder in zip(axes.flat, panel):
        path = rng.choice(folders[folder])
        with Image.open(path) as img:
            rgb = ImageOps.exif_transpose(img).convert("RGB")
            ax.imshow(rgb)
            ax.set_title(f"{folder.name} | {rgb.width}×{rgb.height}", fontsize=8)
    fig.tight_layout()
    plt.show()

In [ ]:
"""Train and evaluate Detective Rice on original microscope images."""
import hashlib
import json
import random
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps
from scipy.fft import dctn
from scipy.spatial.distance import cdist
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

CLASSES = {
    "BD49": ("BRRI dhan49", "ব্রি ধান৪৯"),
    "BD52": ("BRRI dhan52", "ব্রি ধান৫২"),
    "BRRI67": ("BRRI dhan67", "ব্রি ধান৬৭"),
    "BRRI74": ("BRRI dhan74", "ব্রি ধান৭৪"),
    "Binadhan7": ("Binadhan-7", "বিনাধান-৭"),
    "Binadhan20": ("Binadhan-20", "বিনাধান-২০"),
}
SEED = 42
IMAGE_SIZE = 224
EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def read_rgb(path):
    with Image.open(path) as photo:
        return ImageOps.exif_transpose(photo).convert("RGB")


def letterbox(image):
    return ImageOps.pad(image, (IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR, color=(0, 0, 0))


def eval_transform():
    return transforms.Compose([
        transforms.Lambda(letterbox),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ])


def train_transform():
    return transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(12, interpolation=transforms.InterpolationMode.BILINEAR, fill=0),
        transforms.ColorJitter(brightness=0.12, contrast=0.10),
        eval_transform(),
    ])


def make_model(class_count, weights=None):
    model = efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, class_count)
    return model


def perceptual_hash(gray):
    pixels = np.asarray(gray.resize((32, 32), Image.Resampling.LANCZOS), dtype=float)
    coeffs = dctn(pixels, norm="ortho")[:8, :8].ravel()[1:]
    return coeffs > np.median(coeffs)


def fingerprints(path):
    rgb = read_rgb(path)
    digest = hashlib.sha256(rgb.tobytes()).hexdigest()
    gray = rgb.convert("L")
    # The audit shows isolated grains on black. The crop removes surrounding
    # space so resized and recentered copies can be compared as well.
    box = gray.point(lambda value: 255 if value > 24 else 0).getbbox()
    gray = gray.crop(box) if box else gray
    views = [view.rotate(angle, expand=True)
             for view in (gray, ImageOps.mirror(gray))
             for angle in (0, 90, 180, 270)]
    thumbs = [np.asarray(view.resize((48, 48), Image.Resampling.BILINEAR))
              for view in views]
    return digest, [perceptual_hash(view) for view in views], thumbs


def build_split(root, output, group_csv=None):
    rows = [{"path": path.relative_to(root).as_posix(), "class_id": label}
            for label in CLASSES
            for path in sorted((root / label).iterdir())
            if path.suffix.lower() in EXTENSIONS]
    frame = pd.DataFrame(rows)
    counts = frame["class_id"].value_counts().reindex(CLASSES, fill_value=0)
    if (counts < 10).any():
        raise ValueError(f"Too few original files: {counts.to_dict()}")
    parent = list(range(len(frame)))

    def find(index):
        while index != parent[index]:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    def union(a, b):
        parent[find(b)] = find(a)

    if group_csv:
        manifest = pd.read_csv(group_csv, dtype=str, keep_default_na=False)
        if manifest["path"].duplicated().any():
            raise ValueError("Group CSV contains repeated paths.")
        frame = frame.merge(manifest[["path", "group_id"]], how="left", on="path")
        if frame["group_id"].isna().any() or frame["group_id"].str.strip().eq("").any():
            raise ValueError("Provide a group ID for every selected original.")
        for indices in frame.groupby("group_id").indices.values():
            for index in indices[1:]:
                union(indices[0], index)
    else:
        print("No verified physical sample/batch IDs. Split independence is at image-family level only.", flush=True)

    ids_by_digest = {}
    hashes, thumbnails = [], []
    for index, relative in enumerate(frame["path"]):
        digest, variants, thumbs = fingerprints(root / relative)
        hashes.append(variants)
        thumbnails.append(thumbs)
        union(index, ids_by_digest.setdefault(digest, index))
    hashes = np.asarray(hashes, dtype=np.float32)
    thumbnails = np.asarray(thumbnails, dtype=np.uint8)
    # Compare all labels, including mirrored/right-angle variants and recentered
    # crops. Use a conservative six-bit threshold to flag likely image families.
    for start in range(0, len(frame), 128):
        end = min(start + 128, len(frame))
        distances = cdist(hashes[start:end, 0], hashes.reshape(-1, 63), "hamming")
        near = distances.reshape(end - start, len(frame), 8)
        for local, other, orientation in zip(*np.where(near <= 6 / 63)):
            index = start + local
            if index >= other or find(index) == find(other):
                continue
            a = thumbnails[index, 0].astype(np.float32)
            b = thumbnails[other, orientation].astype(np.float32)
            mask = (a > 24) | (b > 24)
            if mask.any() and np.sqrt(np.mean((a[mask] - b[mask]) ** 2)) <= 19:
                union(index, other)
    frame["group"] = [find(index) for index in range(len(frame))]
    if not group_csv:
        conflicting = frame.groupby("group")["class_id"].nunique()
        removed = frame["group"].isin(conflicting[conflicting > 1].index)
        frame.loc[removed].to_csv(output / "cross_label_candidates.csv", index=False)
        print(f"Excluded {removed.sum()} images in cross-label duplicate candidates.", flush=True)
        frame = frame.loc[~removed].reset_index(drop=True)
    print(frame.groupby("class_id")["group"].nunique().to_string(), flush=True)
    if (frame.groupby("class_id")["group"].nunique().reindex(CLASSES, fill_value=0) < 6).any():
        raise ValueError("At least six image families per class are needed. Inspect the duplicate audit and group metadata.")
    folds = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
    frame["fold"] = -1
    for fold, (_, held) in enumerate(folds.split(frame, frame["class_id"], frame["group"])):
        frame.loc[held, "fold"] = fold
    frame["split"] = np.select([frame["fold"].eq(0), frame["fold"].eq(1)], ["test", "validation"], default="train")
    split_counts = pd.crosstab(frame["class_id"], frame["split"]).reindex(CLASSES)
    if set(split_counts.columns) != {"train", "validation", "test"} or (split_counts < 1).any().any():
        raise ValueError(f"Split missing classes: {split_counts}")
    assert frame.groupby("group")["split"].nunique().max() == 1
    frame.to_csv(output / "split_manifest.csv", index=False)
    print(split_counts.to_string(), flush=True)
    return frame


class RiceImages(Dataset):
    def __init__(self, frame, root, preprocess):
        self.paths = [root / path for path in frame["path"]]
        self.labels = [list(CLASSES).index(label) for label in frame["class_id"]]
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, item):
        return self.preprocess(read_rgb(self.paths[item])), self.labels[item]


def loader(frame, root, preprocess, batch_size, shuffle=False):
    return DataLoader(
        RiceImages(frame, root, preprocess), batch_size=batch_size, shuffle=shuffle,
        num_workers=2, pin_memory=torch.cuda.is_available(), persistent_workers=True,
    )


@torch.inference_mode()
def predict(model, batches, device):
    model.eval()
    actual, predicted = [], []
    for images, labels in batches:
        logits = model(images.to(device, non_blocking=True))
        actual.extend(labels.tolist())
        predicted.extend(logits.argmax(dim=1).cpu().tolist())
    return actual, predicted


def report(actual, predicted, elapsed, output, name, class_ids=None):
    class_ids = list(CLASSES) if class_ids is None else class_ids
    numeric = [list(CLASSES).index(label) for label in class_ids]
    names = [CLASSES[label][0] for label in class_ids]
    per_class = classification_report(actual, predicted, labels=numeric, target_names=names,
                                      zero_division=0, output_dict=True)
    result = {"images": len(actual), "accuracy": accuracy_score(actual, predicted),
              "macro_f1": f1_score(actual, predicted, labels=numeric, average="macro", zero_division=0),
              "seconds": elapsed, "classification_report": per_class}
    (output / f"{name}_metrics.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(actual, predicted, labels=numeric, display_labels=names,
                                            cmap="Greens", xticks_rotation=40, ax=ax, colorbar=False)
    fig.tight_layout()
    fig.savefig(output / f"{name}_confusion.png", dpi=155)
    plt.show()
    plt.close(fig)
    print(f"{name}: {len(actual)} images | accuracy={result['accuracy']:.4f} | macro F1={result['macro_f1']:.4f} | {elapsed:.1f}s", flush=True)
    print(pd.DataFrame(per_class).T.reindex(names).to_string(), flush=True)
    return result


def evaluate_phone(model, device, phone_csv, output, batch_size):
    phone = pd.read_csv(phone_csv, dtype=str, keep_default_na=False)
    fields = ["path", "class_id", "batch_id", "condition", "provenance"]
    if phone.empty or not set(fields).issubset(phone) or (phone[fields].apply(lambda col: col.str.strip()) == "").any().any():
        raise ValueError("Phone CSV needs populated path, class_id, batch_id, condition and provenance columns.")
    if phone["path"].duplicated().any() or set(phone["class_id"]) != set(CLASSES):
        raise ValueError("Phone CSV needs unique image paths and all six class IDs.")
    start = time.perf_counter()
    actual, predicted = predict(model, loader(phone, phone_csv.parent, eval_transform(), batch_size), device)
    phone["prediction"] = [list(CLASSES)[index] for index in predicted]
    phone["correct"] = phone["class_id"].eq(phone["prediction"])
    phone.to_csv(output / "phone_predictions.csv", index=False)
    summary = phone.groupby(["class_id", "batch_id", "condition"])["correct"].agg(["size", "mean"])
    summary.to_csv(output / "phone_accuracy_by_batch_and_condition.csv")
    for condition, subset in phone.groupby("condition"):
        labels = [list(CLASSES).index(label) for label in sorted(subset["class_id"].unique())]
        reference = subset["class_id"].map(list(CLASSES).index)
        guesses = subset["prediction"].map(list(CLASSES).index)
        print(f"{condition}: n={len(subset)}, accuracy={subset['correct'].mean():.3f}, "
              f"macro F1 over {len(labels)} represented classes={f1_score(reference, guesses, labels=labels, average='macro', zero_division=0):.3f}")
    return report(actual, predicted, time.perf_counter() - start, output, "phone")


def run(root, output, *, epochs=10, batch_size=32, patience=3, group_csv=None, phone_csv=None):
    output.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Kaggle GPU accelerator or run on a CUDA-enabled desktop.")
    device = torch.device("cuda")
    print(torch.cuda.get_device_name(device), flush=True)
    model = make_model(len(CLASSES), EfficientNet_B0_Weights.IMAGENET1K_V1).to(device)
    frame = build_split(root, output, group_csv)
    train = frame[frame["split"].eq("train")]
    validation = frame[frame["split"].eq("validation")]
    test = frame[frame["split"].eq("test")]
    # Compare a training original with the exact input framing at 224 pixels.
    original = read_rgb(root / train.iloc[0]["path"])
    fig, axes = plt.subplots(1, 2, figsize=(7, 3))
    for ax, image, title in zip(axes, [original, letterbox(original)], ["Original RGB", "224×224 model framing"]):
        ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(output / "resize_check.png", dpi=150)
    plt.show()
    plt.close(fig)

    train_loader = loader(train, root, train_transform(), batch_size, shuffle=True)
    validation_loader = loader(validation, root, eval_transform(), batch_size)
    optimizer = torch.optim.AdamW([
        {"params": model.features.parameters(), "lr": 0.00015},
        {"params": model.classifier.parameters(), "lr": 0.001},
    ], weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda")
    loss_fn = nn.CrossEntropyLoss()
    best_f1, stale, history = -1.0, 0, []
    checkpoint = output / "detective_rice.pt"
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        total_loss = torch.zeros((), device=device)
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16):
                loss = loss_fn(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.detach() * len(labels)
        actual, predicted = predict(model, validation_loader, device)
        score = f1_score(actual, predicted, labels=range(len(CLASSES)), average="macro", zero_division=0)
        history.append({"epoch": epoch, "loss": total_loss.item() / len(train),
                        "val_macro_f1": score, "val_accuracy": accuracy_score(actual, predicted),
                        "train_and_validation_seconds": round(time.perf_counter() - start, 2)})
        pd.DataFrame(history).to_csv(output / "epoch_log.csv", index=False)
        print(history[-1], flush=True)
        if score > best_f1:
            torch.save({"weights": model.state_dict(), "class_ids": list(CLASSES),
                        "epoch": epoch, "val_macro_f1": score}, checkpoint)
            best_f1, stale = score, 0
        else:
            stale += 1
            if stale >= patience:
                break
    saved = torch.load(checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(saved["weights"])
    print(f"Best epoch: {saved['epoch']} | validation macro F1={saved['val_macro_f1']:.4f}")
    start = time.perf_counter()
    actual, predicted = predict(model, loader(test, root, eval_transform(), batch_size), device)
    report(actual, predicted, time.perf_counter() - start, output, "held_out")
    if phone_csv:
        evaluate_phone(model, device, Path(phone_csv), output, batch_size)
    else:
        print("Fresh phone evaluation pending labeled photos.")
    return checkpoint

roots = list(Path("/kaggle/input").rglob("Original/Original"))
original_root = next(p for p in roots if "rice-varieties-in-bangladesh" in str(p))
output = Path("/kaggle/working/detective_rice")
checkpoint = run(original_root, output, epochs=10)


In [ ]:
# Optional fresh phone-photo evaluation. Attach the images and CSV as a separate Kaggle input.
# CSV columns: path,class_id,batch_id,condition,provenance
# Use paths relative to the location of the CSV and exact dataset class IDs.
PHONE_MANIFEST = None  # Replace with the Path to your labeled phone manifest.
if PHONE_MANIFEST is not None:
    phone_checkpoint = Path("/kaggle/working/detective_rice/detective_rice.pt")
    phone_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    phone_saved = torch.load(phone_checkpoint, map_location=phone_device, weights_only=True)
    phone_model = make_model(len(CLASSES)).to(phone_device)
    phone_model.load_state_dict(phone_saved["weights"])
    evaluate_phone(phone_model, phone_device, Path(PHONE_MANIFEST), phone_checkpoint.parent, 32)
else:
    print("Fresh phone-photo evaluation awaits the labeled phone manifest.")


# Live demo. Run this cell after the held-out evaluation finishes.
import gradio as gr

demo_checkpoint = Path("/kaggle/working/detective_rice/detective_rice.pt")
saved_demo = torch.load(demo_checkpoint, map_location="cpu", weights_only=True)
demo_labels = saved_demo["class_ids"]
demo_model = make_model(len(demo_labels))
demo_model.load_state_dict(saved_demo["weights"])
demo_model.eval()

@torch.inference_mode()
def compare_grain(photo):
    if photo is None:
        raise gr.Error("Upload a photo of one uncooked grain.")
    rgb = ImageOps.exif_transpose(photo).convert("RGB")
    scores = demo_model(eval_transform()(rgb).unsqueeze(0)).softmax(1)[0].tolist()
    return {f"{CLASSES[label][0]} · {CLASSES[label][1]}": score
            for label, score in zip(demo_labels, scores)}, letterbox(rgb)

manifest = pd.read_csv(demo_checkpoint.parent / "split_manifest.csv")
smoke_path = original_root / manifest.loc[manifest["split"].eq("test"), "path"].iloc[0]
smoke_scores, smoke_preview = compare_grain(read_rgb(smoke_path))
assert len(smoke_scores) == 6 and smoke_preview.size == (224, 224)
print("Inference check passed on a held-out original.", flush=True)

demo = gr.Interface(
    fn=compare_grain,
    inputs=gr.Image(type="pil", sources=["upload"], label="Uncooked rice grain"),
    outputs=[gr.Label(num_top_classes=3, label="Closest matches"),
             gr.Image(type="pil", label="224 × 224 model view")],
    title="Detective Rice 😼",
    description=("Compare one grain against six Bangladeshi varieties. "
                 "Use a sharp, tight close-up on a matte black background. "
                 "The model was trained on microscope images. Its scores compare the six supported classes. "
                 "Phone-photo performance is pending validation."),
    analytics_enabled=False,
)
demo.launch(share=True, inline=False, prevent_thread_lock=True, show_error=False)


In [ ]:
# Twelve full-resolution originals for testing the trained classifier.
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from io import StringIO
import csv
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from scipy.ndimage import binary_erosion, gaussian_filter, laplace
from IPython.display import display, FileLink

preview_root = Path("/kaggle/working")
preview_originals = next(p for p in Path("/kaggle/input").rglob("Original/Original")
                         if "rice-varieties-in-bangladesh" in str(p))
preview_manifest = pd.read_csv(preview_root / "detective_rice" / "split_manifest.csv")
preview_labels = {"BD49": "BRRI_dhan49", "BD52": "BRRI_dhan52", "BRRI67": "BRRI_dhan67",
                  "BRRI74": "BRRI_dhan74", "Binadhan7": "Binadhan-7", "Binadhan20": "Binadhan-20"}

def preview_score(path):
    with Image.open(path) as image:
        rgb = ImageOps.exif_transpose(image).convert("RGB")
    if rgb.size != (640, 480):
        return -1
    gray = np.asarray(rgb.convert("L"), dtype=np.float32)
    grain = gray > 37
    area = binary_erosion(grain, iterations=3)
    if area.sum() < 2000 or grain[:8].any() or grain[-8:].any() or grain[:, :8].any() or grain[:, -8:].any():
        return -1
    return float(laplace(gaussian_filter(gray, 0.8))[area].var())

demo_zip = preview_root / "Detective_Rice_12_Test_Images.zip"
gallery = []
details = StringIO()
writer = csv.writer(details)
writer.writerow(["filename", "class_id", "original_dataset_path", "original_split", "group", "width", "height"])
with ZipFile(demo_zip, "w", ZIP_DEFLATED) as archive:
    for label, friendly in preview_labels.items():
        candidates = preview_manifest[(preview_manifest["split"] == "test") &
                                      (preview_manifest["class_id"] == label)].copy()
        candidates["detail_score"] = [preview_score(preview_originals / name) for name in candidates["path"]]
        chosen = (candidates[candidates["detail_score"] >= 0].sort_values("detail_score", ascending=False)
                  .drop_duplicates("group").head(2))
        assert len(chosen) == 2, label
        for number, row in enumerate(chosen.itertuples(index=False), 1):
            name = f"{friendly}_{number:02d}.jpg"
            source = preview_originals / row.path
            archive.write(source, name)
            writer.writerow([name, label, row.path, "test", row.group, 640, 480])
            gallery.append((name, source))
    archive.writestr("labels.csv", details.getvalue())
    archive.writestr("README.txt", "Twelve original 640 x 480 JPEGs, two per class, selected from the recorded held-out test split. Images are original bytes. These are demo checks, not an independent dataset or fresh phone photos. Source: Md Tahsin et al., DOI 10.17632/2fgv99854n.1, Kaggle mirror fahadbd/rice-varieties-in-bangladesh. Follow the dataset attribution and license terms.\n")

with ZipFile(demo_zip) as archive:
    assert archive.testzip() is None and len(archive.namelist()) == 14
print(f"Verified {len(gallery)} original JPGs: {demo_zip.name} ({demo_zip.stat().st_size:,} bytes)")
for name, source in gallery:
    print(name, "<-", source.relative_to(preview_originals))
display(FileLink(demo_zip.name))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 3, figsize=(12, 11), layout="constrained")
for axis, (name, source) in zip(axes.flat, gallery):
    with Image.open(source) as photo:
        axis.imshow(photo)
    axis.set_title(name, fontsize=9)
    axis.axis("off")
plt.show()


In [ ]:
APP_FILES = {"requirements-pytorch.txt": "--index-url https://download.pytorch.org/whl/cpu\ntorch==2.7.1\ntorchvision==0.22.1\n", "rice_inference.py": "\"\"\"Shared preprocessing and CPU inference for Detective Rice.\"\"\"\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image, ImageOps\n\nROOT = Path(__file__).resolve().parent\n\n\ndef read_metadata(folder=ROOT):\n    return json.loads((folder / \"model_metadata.json\").read_text(encoding=\"utf-8\"))\n\n\ndef preprocess(photo, metadata):\n    settings = metadata[\"preprocessing\"]\n    rgb = ImageOps.exif_transpose(photo).convert(\"RGB\")\n    view = ImageOps.pad(\n        rgb, tuple(settings[\"size\"]), Image.Resampling.BILINEAR,\n        color=tuple(settings[\"padding_rgb\"]), centering=tuple(settings[\"centering\"]),\n    )\n    pixels = np.asarray(view, dtype=np.float32) / np.float32(255)\n    mean = np.array(settings[\"mean\"], dtype=np.float32)\n    std = np.array(settings[\"std\"], dtype=np.float32)\n    tensor = np.ascontiguousarray(((pixels - mean) / std).transpose(2, 0, 1))\n    return tensor, view\n\n\ndef pytorch_model(class_count):\n    from torch import nn\n    from torchvision.models import efficientnet_b0\n\n    model = efficientnet_b0(weights=None)\n    model.classifier[1] = nn.Linear(model.classifier[1].in_features, class_count)\n    return model\n\n\ndef load_predictor(backend, metadata, folder=ROOT):\n    if backend == \"ONNX Runtime\":\n        import onnxruntime as ort\n\n        options = ort.SessionOptions()\n        options.intra_op_num_threads = 2\n        session = ort.InferenceSession(\n            str(folder / \"rice_variety_model.onnx\"), sess_options=options,\n            providers=[\"CPUExecutionProvider\"],\n        )\n        embedded = session.get_modelmeta().custom_metadata_map.get(\"detective_rice\")\n        if embedded is None or json.loads(embedded) != metadata:\n            raise ValueError(\"The ONNX file and model_metadata.json are from different exports.\")\n\n        def predict(batch):\n            return session.run([\"logits\"], {\"images\": batch})[0]\n\n        return predict\n\n    if backend == \"PyTorch\":\n        import torch\n\n        torch.set_num_threads(2)\n        saved = torch.load(folder / \"rice_variety_model.pth\", map_location=\"cpu\", weights_only=True)\n        if saved[\"metadata\"] != metadata:\n            raise ValueError(\"The PyTorch file and model_metadata.json are from different exports.\")\n        model = pytorch_model(len(metadata[\"classes\"]))\n        model.load_state_dict(saved[\"state_dict\"])\n        model.eval()\n\n        @torch.inference_mode()\n        def predict(batch):\n            return model(torch.from_numpy(batch)).numpy()\n\n        return predict\n\n    raise ValueError(f\"Unknown backend: {backend}\")\n\n\ndef scores(logits):\n    weights = np.exp(logits - np.max(logits))\n    return weights / weights.sum()\n", "README.md": "# Detective Rice 😼\n\nThe bundle includes the trained EfficientNet-B0 model in two formats, a shared preprocessing implementation, a local Streamlit app, and parity reports. The checkpoint is the Kaggle run's epoch 3 model, selected by validation macro F1.\n\n## Run locally on Windows\n\nUse 64-bit Python 3.11 or 3.12. Unzip the bundle, open PowerShell in the extracted folder, and run:\n\n```powershell\npy -3.12 -m venv .venv\n.\\.venv\\Scripts\\python.exe -m pip install -r requirements-pytorch.txt\n.\\.venv\\Scripts\\python.exe -m pip install -r requirements.txt\n.\\.venv\\Scripts\\python.exe -m streamlit run app.py\n```\n\nUse `-3.11` in the first line if that is your installed version. On macOS, use `python3 -m venv .venv`, install `torch==2.7.1 torchvision==0.22.1` from PyPI with `.venv/bin/python -m pip install`, then install `requirements.txt` and launch Streamlit using `.venv/bin/python`. On Linux the CPU requirements file works with the same `.venv/bin/python` executable. The app defaults to PyTorch and includes an ONNX Runtime switch.\n\n## Model files\n\n- `rice_variety_model.pth` contains a CPU state dictionary under `state_dict` and configuration under `metadata`. Load with `weights_only=True`.\n- `rice_variety_model.onnx` accepts float32 `images` with shape `[N,3,224,224]` and returns float32 `logits` with shape `[N,6]`. Batch size is dynamic. Apply softmax across the six logits for display.\n- `model_metadata.json` records output order, the data type and tensor layout, and the exact image preprocessing settings. It is also embedded in both model files.\n- `parity_report.json` and `parity_predictions.csv` record the comparison on the original, fixed held-out split. `sha256.json` records file checksums.\n\nThe model expects a single, uncooked, dehusked grain. The original training and held-out data were microscope photos. Fresh phone-photo accuracy remains unmeasured.\n", "requirements.txt": "streamlit>=1.45,<2\nonnxruntime>=1.19,<2\nnumpy>=1.26,<3\nPillow>=10.4,<13\n", "app.py": "from PIL import Image, UnidentifiedImageError\nimport streamlit as st\n\nfrom rice_inference import load_predictor, preprocess, read_metadata, scores\n\n\nst.set_page_config(page_title=\"Detective Rice\", page_icon=\"😼\")\nst.title(\"Detective Rice 😼\")\nst.caption(\"Bangladeshi rice variety recognition\")\nst.write(\"Upload a sharp close-up of one uncooked, dehusked grain against a matte black background.\")\nmetadata = read_metadata()\nbackend = st.radio(\"Inference engine\", [\"PyTorch\", \"ONNX Runtime\"], horizontal=True)\nwith st.expander(\"Supported varieties\"):\n    for label in metadata[\"classes\"]:\n        st.write(f'{label[\"name_en\"]} · {label[\"name_bn\"]}')\n\n\n@st.cache_resource\ndef cached_predictor(selected_backend):\n    return load_predictor(selected_backend, read_metadata())\n\n\nupload = st.file_uploader(\"Grain photo\", type=[\"jpg\", \"jpeg\", \"png\", \"webp\"])\nif upload:\n    try:\n        with Image.open(upload) as photo:\n            tensor, preview = preprocess(photo, metadata)\n    except (UnidentifiedImageError, OSError, Image.DecompressionBombError):\n        st.error(\"Could not open the image. Try a standard JPEG or PNG.\")\n        st.stop()\n    try:\n        probabilities = scores(cached_predictor(backend)(tensor[None])[0])\n    except (ImportError, OSError, ValueError, RuntimeError) as exc:\n        st.error(f\"Could not run {backend}. {exc}\")\n        st.stop()\n    left, right = st.columns([1, 1], gap=\"large\")\n    with left:\n        st.image(preview, width=224, caption=\"224 × 224 model input\")\n    with right:\n        st.subheader(\"Closest matches\")\n        for index in probabilities.argsort()[::-1][:3]:\n            label = metadata[\"classes\"][index]\n            probability = float(probabilities[index])\n            st.write(f'**{label[\"name_en\"]}** · {label[\"name_bn\"]}')\n            st.progress(probability, text=f\"Model score {probability:.1%}\")\n    st.caption(\"Trained on microscope images. Scores compare the six supported varieties. Fresh phone-photo performance is unmeasured.\")\n"}
# Run after training, or with the original Kaggle checkpoint and manifest restored.
from pathlib import Path
import csv
import hashlib
import importlib
import importlib.util
import json
import subprocess
import sys
import time
from zipfile import ZIP_DEFLATED, ZipFile

for package in ("onnx", "onnxruntime", "streamlit"):
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        importlib.invalidate_caches()

import numpy as np
import onnx
import onnxruntime
import pandas as pd
from PIL import Image
import torch
import torchvision

working = Path("/kaggle/working")
baseline = working / "detective_rice"
export_dir = working / "Detective_Rice_Local"
export_dir.mkdir(exist_ok=True)
for name, contents in APP_FILES.items():
    (export_dir / name).write_text(contents, encoding="utf-8")
if str(export_dir) not in sys.path:
    sys.path.insert(0, str(export_dir))
from rice_inference import load_predictor, preprocess, pytorch_model

torch.set_num_threads(2)
checkpoint = torch.load(baseline / "detective_rice.pt", map_location="cpu", weights_only=True)
assert checkpoint["epoch"] == 3, "Expected the selected checkpoint from the baseline."
assert checkpoint["class_ids"] == list(CLASSES), "Unexpected class order."
split_path = baseline / "split_manifest.csv"
metadata = {
    "architecture": "torchvision EfficientNet-B0",
    "architecture_id": "efficientnet_b0",
    "checkpoint_epoch": checkpoint["epoch"],
    "validation_macro_f1": checkpoint["val_macro_f1"],
    "checkpoint_source": "Kaggle Detective Rice, run 374163214",
    "classes": [
        {"index": index, "dataset_id": class_id, "name_en": CLASSES[class_id][0],
         "name_bn": CLASSES[class_id][1]}
        for index, class_id in enumerate(checkpoint["class_ids"])
    ],
    "preprocessing": {
        "size": [224, 224], "color_mode": "RGB", "respect_exif_orientation": True,
        "resize": "aspect-preserving resize with centered padding; no crop",
        "interpolation": "PIL bilinear", "padding_rgb": [0, 0, 0], "centering": [0.5, 0.5],
        "pixel_scale_divisor": 255, "mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225],
    },
    "input": {"name": "images", "dtype": "float32", "layout": "NCHW", "shape": ["batch", 3, 224, 224]},
    "output": {"name": "logits", "dtype": "float32", "shape": ["batch", 6]},
    "onnx_opset": 17,
}
(export_dir / "model_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
pth = export_dir / "rice_variety_model.pth"
onnx_file = export_dir / "rice_variety_model.onnx"
torch.save({"state_dict": checkpoint["weights"], "metadata": metadata}, pth)
model = pytorch_model(len(metadata["classes"]))
model.load_state_dict(checkpoint["weights"])
model.eval()
with torch.inference_mode():
    torch.onnx.export(
        model, torch.zeros((1, 3, 224, 224), dtype=torch.float32), str(onnx_file),
        input_names=["images"], output_names=["logits"], opset_version=17, dynamo=False,
        dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
    )
graph = onnx.load(onnx_file)
entry = graph.metadata_props.add()
entry.key = "detective_rice"
entry.value = json.dumps(metadata, ensure_ascii=False)
onnx.save(graph, onnx_file)
onnx.checker.check_model(str(onnx_file), full_check=True)
print("Both export files created. Starting CPU parity on all held-out images.", flush=True)

test = pd.read_csv(split_path).query("split == 'test'").reset_index(drop=True)
assert len(test) == 495
dataset_root = next(p for p in Path("/kaggle/input").rglob("Original/Original")
                    if "rice-varieties-in-bangladesh" in str(p))
predict_torch = load_predictor("PyTorch", metadata, export_dir)
predict_ort = load_predictor("ONNX Runtime", metadata, export_dir)
baseline_transform = eval_transform()
class_ids = checkpoint["class_ids"]
fields = ["path", "true_class_id", "pytorch_top1", "onnx_top1", "top1_match", "top3_order_match", "max_abs_logit_difference"]
rows = []
max_input_delta = 0
absolute_differences = []
start = time.perf_counter()
for offset in range(0, len(test), 16):
    examples = test.iloc[offset:offset+16]
    pixels = []
    for relative in examples["path"]:
        path = dataset_root / relative
        with Image.open(path) as photo:
            input_tensor, _ = preprocess(photo, metadata)
        original_tensor = baseline_transform(read_rgb(path)).numpy()
        max_input_delta = max(max_input_delta, float(np.max(np.abs(original_tensor - input_tensor))))
        pixels.append(input_tensor)
    batch = np.stack(pixels)
    torch_logits = predict_torch(batch)
    onnx_logits = predict_ort(batch)
    assert torch_logits.shape == onnx_logits.shape == (len(batch), len(class_ids))
    error = np.abs(torch_logits - onnx_logits)
    absolute_differences.append(error)
    torch_order = np.argsort(-torch_logits, axis=1)
    onnx_order = np.argsort(-onnx_logits, axis=1)
    for row, a, b, delta in zip(examples.itertuples(index=False), torch_order, onnx_order, error):
        rows.append({"path": row.path, "true_class_id": row.class_id,
                     "pytorch_top1": class_ids[a[0]], "onnx_top1": class_ids[b[0]],
                     "top1_match": bool(a[0] == b[0]), "top3_order_match": bool(np.array_equal(a[:3], b[:3])),
                     "max_abs_logit_difference": float(delta.max())})
    if (offset // 16 + 1) % 8 == 0 or offset + len(batch) == len(test):
        print(f"Compared {offset + len(batch)} / {len(test)} images.", flush=True)
with (export_dir / "parity_predictions.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    writer.writerows(rows)

# Test the graph's declared single-image route separately from batched parity.
assert predict_torch(batch[:1]).shape == predict_ort(batch[:1]).shape == (1, len(class_ids))
single_torch, single_ort = predict_torch(batch[:1]), predict_ort(batch[:1])
assert int(single_torch.argmax()) == int(single_ort.argmax())
assert np.allclose(single_torch, single_ort, atol=5e-3, rtol=0)
elapsed = time.perf_counter() - start
diff = np.concatenate(absolute_differences)
report = {
    "held_out_images": len(rows),
    "matching_top1": sum(row["top1_match"] for row in rows),
    "matching_ordered_top3": sum(row["top3_order_match"] for row in rows),
    "pytorch_correct": sum(row["pytorch_top1"] == row["true_class_id"] for row in rows),
    "onnx_correct": sum(row["onnx_top1"] == row["true_class_id"] for row in rows),
    "max_abs_logit_difference": float(diff.max()),
    "mean_abs_logit_difference": float(diff.mean()),
    "max_preprocessing_delta_vs_original_pytorch_pipeline": max_input_delta,
    "absolute_logit_tolerance": 0.005,
    "all_logits_within_absolute_tolerance": bool(diff.max() <= 0.005),
    "onnx_runtime_provider": "CPUExecutionProvider",
    "single_image_dynamic_batch_verified": True,
    "verification_seconds": round(elapsed, 3),
    "versions": {"pytorch": torch.__version__, "torchvision": torchvision.__version__,
                 "onnx": onnx.__version__, "onnxruntime": onnxruntime.__version__},
    "split_manifest_sha256": hashlib.sha256(split_path.read_bytes()).hexdigest(),
}
assert max_input_delta == 0.0, report
assert report["matching_top1"] == len(rows), report
assert report["matching_ordered_top3"] == len(rows), report
assert report["all_logits_within_absolute_tolerance"], report

from streamlit.testing.v1 import AppTest
app_test = AppTest.from_file(str(export_dir / "app.py")).run(timeout=25)
assert len(app_test.exception) == 0
assert app_test.title[0].value == "Detective Rice 😼"
app_test.radio[0].set_value("ONNX Runtime").run(timeout=25)
assert len(app_test.exception) == 0 and app_test.radio[0].value == "ONNX Runtime"
report["streamlit_startup_and_engine_switch_smoke_test"] = "passed"
(export_dir / "parity_report.json").write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(report, indent=2), flush=True)

checksums = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
             for path in sorted(export_dir.iterdir()) if path.is_file() and path.name != "sha256.json"}
(export_dir / "sha256.json").write_text(json.dumps(checksums, indent=2) + "\n", encoding="utf-8")
bundle = working / "Detective_Rice_Local_Models_and_App.zip"
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(export_dir.iterdir()):
        if path.is_file():
            archive.write(path, path.name)
with ZipFile(bundle) as archive:
    assert archive.testzip() is None and {"rice_variety_model.pth", "rice_variety_model.onnx", "app.py"} <= set(archive.namelist())
print(f"Verified {bundle.name}: {bundle.stat().st_size:,} bytes", flush=True)
from IPython.display import FileLink, display
display(FileLink(bundle.name))



In [ ]:
print("top1 match", sum(row["top1_match"] for row in rows), "top3 exact",sum(row["top3_order_match"] for row in rows))
print("max logit diff",np.concatenate(absolute_differences).max(),"max preproc",max_input_delta, "normalized tolerance",max_relative_tolerance)
print("single PyTorch", predict_torch(batch[:1]))
print("single ONNX", predict_ort(batch[:1]))
print("same-input batch first", predict_ort(batch)[0])
